In [33]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

import os
from rag_helper import RAGHelper
from ingest import load_faq_data
from minsearch import Index



In [34]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [44]:
print(len(documents))

chunks = chunk_documents(documents)
print(len(chunks))
import json

index = Index(
  text_fields=['content'],
  keyword_fields=['filename']
)

index.fit(documents)
# format the output of index.search('How does the agentic loop keep calling the model until it stops?'))
result = index.search('How does the agentic loop keep calling the model until it stops?')
print(json.dumps(result, indent=2))


chunk_index = Index(
  text_fields=['content'],
  keyword_fields=['filename']
)

chunk_index.fit(chunks)
print(f'chunk index length: {len(chunk_index.docs)}')


72
295
[
  {
    "content": "# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don't know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it's\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver's seat, we have an agent. It's an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can ca

In [62]:
class new_rag_helper(RAGHelper):
  def search(self, query: str, num_results: int = 5) -> dict[str, str]:
    """
    Search the index for the query.
    """
    result = self.index.search(query)
    return result

  def build_prompt(self, query, search_results):
    prompt = self.user_prompt_template.format(
      user_question=query, 
      context=search_results
    )
    return prompt.strip()

  def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)

    usage = answer.usage
    return (answer.output_text, usage)

rag_helper = new_rag_helper(
  index, 
  openai_client,
  instructions="""
  You are a helpful assistant that can answer questions about the course given the provided context.
  Use the context to find relevant information and provide accurate answers.
  If an answer is not found in the context, respond with "I don't know"
  """
)

search_results = rag_helper.search('How does the agentic loop keep calling the model until it stops?')
print(search_results)
#print(json.dumps(rag_helper.build_prompt('How does the agentic loop keep calling the model until it stops?', search_results), indent=2))

#answer = rag_helper.rag('How does the agentic loop keep calling the model until it stops?')







[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [ ]:
#print(answer[1])


ResponseUsage(input_tokens=11561, input_tokens_details=InputTokensDetails(cached_tokens=11008, cache_write_tokens=0), output_tokens=225, output_tokens_details=OutputTokensDetails(reasoning_tokens=62), total_tokens=11786)


In [55]:
chunked_rag_helper = new_rag_helper(
  chunk_index, 
  openai_client,
  instructions="""
  You are a helpful assistant that can answer questions about the course given the provided context.
  Use the context to find relevant information and provide accurate answers.
  If an answer is not found in the context, respond with "I don't know"
  """
)


chunked_answer = chunked_rag_helper.rag('How does the agentic loop keep calling the model until it stops?')


In [56]:
print(chunked_answer[1])
print(chunked_answer[0])

ResponseUsage(input_tokens=5470, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=222, output_tokens_details=OutputTokensDetails(reasoning_tokens=52), total_tokens=5692)
It uses a `while True` loop with a flag like `has_function_calls`.

On each iteration, it:

1. Calls the model with the current `messages`
2. Appends `response.output` to the message history
3. Checks each output item:
   - if it’s a `function_call`, the code runs the tool, appends the tool result back to `messages`, and sets `has_function_calls = True`
   - if it’s a normal `message`, it prints or stores the assistant’s answer
4. At the end of the iteration, it breaks only if `has_function_calls == False`

So the loop keeps going as long as the model asks for tools, and it stops when the model returns a response with no function calls — meaning it has produced its final answer.


In [63]:
# agentic loop

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [65]:
agent_tools = Tools()
# if you define the tool with docstring and types, then add_tool can infer
# craft the tool from the docstring
agent_tools.add_tool(chunked_rag_helper.search)
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
  tools=agent_tools,
  developer_prompt="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.",
  chat_interface=chat_interface,
  llm_client=OpenAIClient(model='gpt-5.4-mini')
)



In [66]:
result = runner.loop(
  prompt='How does the agentic loop work, and how is it different from plain RAG?',
  callback=callback
)

result.cost
result.all_messages

-> Response received


-> Response received


[EasyInputMessage(content="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop vs RAG iterative retrieval generation tool use explain","num_results":"5"}', call_id='call_r19utSwczXya1AvslUTaKFUW', name='search', type='function_call', id='fc_07e85e1d50060290006a5fe088e2248194a6b450337071cda3', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"plain RAG definition retrieval augmented generation one-shot retrieval generation","num_results":"5"}', call_id='call_9aK1sMMILHdH8hLs3mKHyUpb', name='search', type='function_call', id='fc_07e85e1d50060290006a5fe088e23c8194b80e381e3bed8429', namespace=None, status='completed'),

In [67]:
print(result.cost)

CostInfo(input_cost=Decimal('0.0118725'), output_cost=Decimal('0.0029475'), total_cost=Decimal('0.0148200'))
